[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/04_batch_size_memory/04_batch_size_memory.ipynb)

# 04. Batch Size, Memory & Mixed Precision

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/04_batch_size_memory')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '04_batch_size_memory':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '04_batch_size_memory'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 1. Batch Size & Gradient Variance

$$
\text{Var}[\nabla \mathcal{L}_B] = \frac{\sigma^2}{B}
$$

Small batch → high variance → noise helps escape sharp minima. Large batch → low variance → may need LR scaling.


## 2. Linear Scaling Rule

$$
\eta_{new} = \eta_{base} \cdot \frac{B_{new}}{B_{base}}
$$

Goyal et al. (2017): when batch 256→8192, scale LR linearly + warmup. Breaks when batch too large without LARS/LAMB.


## 3. Gradient Accumulation

$$
g_{accum} = \frac{1}{K}\sum_{k=1}^{K} g_k
$$

Mathematically equivalent to batch size $B \times K$ without storing all activations at once.


In [ ]:
def simulate_batch_variance(Bs, sigma2=1.0, trials=500):
    means = []
    for B in Bs:
        vars_ = [np.var(np.random.randn(B) * np.sqrt(sigma2/B)) for _ in range(trials)]
        means.append(np.mean(vars_))
    return means

Bs = [1, 4, 16, 64, 256]
plt.plot(Bs, simulate_batch_variance(Bs), 'o-')
plt.xlabel('Batch size B'); plt.ylabel('Var[grad estimate]'); plt.title('Variance ~ 1/B'); plt.xscale('log'); plt.show()


## 4. GPU Memory Breakdown

$$
\text{GPU\_Mem} = \text{Params} + \text{Optimizer} + \text{Activations} + \text{Gradients}
$$

- Params: $P \times \text{bytes}$
- Adam states: $3P$ (params + $m$ + $v$)
- Activations: $\propto B \times L \times d^2$


In [ ]:
def estimate_memory(P, B, L, d, bytes_per=4, optimizer='adam'):
    params = P * bytes_per
    grads = P * bytes_per
    opt = 3 * P * bytes_per if optimizer == 'adam' else 2 * P * bytes_per
    act = B * L * d * d * bytes_per  # simplified per-layer
    total = params + grads + opt + act
    return {'params_mb': params/1e6, 'grads_mb': grads/1e6, 'opt_mb': opt/1e6, 'act_mb': act/1e6, 'total_mb': total/1e6}

P = 125_000_000  # ~125M params
est = estimate_memory(P, B=8, L=24, d=1024)
for k, v in est.items(): print(f'{k}: {v:.1f} MB')


## 5. Mixed Precision — FP16 vs BF16

| Format | Exponent | Mantissa | Range |
|--------|----------|----------|-------|
| FP32 | 8 bit | 23 bit | ±3.4e38 |
| FP16 | 5 bit | 10 bit | ±65504 |
| BF16 | 8 bit | 7 bit | ±3.4e38 |

Loss scaling compensates for FP16 grad underflow.


In [ ]:
# Gradient accumulation demo
model = nn.Linear(10, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
accum_steps = 4
opt.zero_grad()
for k in range(accum_steps):
    x = torch.randn(8, 10)
    loss = model(x).sum() / accum_steps
    loss.backward()
opt.step()
print('Gradient accumulation: 4 micro-batches of 8 = effective batch 32')


In [ ]:
# AMP training loop sketch
from torch.cuda.amp import autocast, GradScaler
use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)
model = nn.Linear(10, 2); opt = torch.optim.Adam(model.parameters())
x = torch.randn(16, 10)
opt.zero_grad()
with autocast(enabled=use_amp):
    loss = F.cross_entropy(model(x), torch.randint(0, 2, (16,)))
scaler.scale(loss).backward()
scaler.step(opt); scaler.update()
print(f'AMP step completed (cuda={use_amp})')


## 6. Linear Scaling Numerical Example

Base: $B=256$, $\eta=0.1$. Scale to $B=2048$ (8×):

$$
\eta_{new} = 0.1 \times \frac{2048}{256} = 0.8
$$

With 500-step linear warmup from 0 → 0.8.


In [ ]:
# Linear scaling rule demo
eta_base, B_base = 0.1, 256
for B_new in [256, 512, 1024, 2048, 8192]:
    eta = eta_base * B_new / B_base
    print(f'B={B_new:5d} -> eta={eta:.3f}')


## 7. ZeRO Stages

| Stage | Sharded | Memory Reduction |
|-------|---------|------------------|
| 0 | Nothing (DDP) | 1× |
| 1 | Optimizer states | ~4× on optimizer |
| 2 | + Gradients | ~8× total |
| 3 | + Parameters | Linear in GPU count |


## 8. OOM Solutions

| Technique | Memory Saved | Compute Cost |
|-----------|---------------|--------------|
| Grad accumulation | Activations ↓ | Same total |
| Checkpointing | $O(L) \to O(\sqrt{L})$ | ~33% extra |
| ZeRO Stage 1 | Optimizer sharded | Communication |
| ZeRO Stage 3 | All states sharded | More comm |
| FSDP | Like ZeRO-3 in PyTorch | Built-in |


## References & Further Reading

- Goyal et al. (2017) — Accurate, Large Minibatch SGD — [arXiv:1706.02677](https://arxiv.org/abs/1706.02677)
- Micikevicius et al. (2018) — Mixed Precision Training — [arXiv:1710.03740](https://arxiv.org/abs/1710.03740)
- Rajbhandari et al. (2020) — ZeRO — [arXiv:1910.02054](https://arxiv.org/abs/1910.02054)
